# Harmonization: NSL-KDD → ตารางกลาง

Notebook นี้แสดงการแปลงข้อมูล NSL-KDD ที่ผ่าน cleansing แล้วให้ตรงกับ schema กลาง โค้ดสำหรับชุดข้อมูลนี้อยู่ใน notebook โดยตรง ไม่ต้องเปิดไฟล์ฟังก์ชันอื่นเพื่อดูว่าแต่ละคอลัมน์ถูกสร้างอย่างไร

ผลลัพธ์มี 3 ตารางที่เชื่อมด้วย record_id:
- features: ค่าที่อาจใช้เป็น input ของโมเดล
- labels: คำตอบจริงสำหรับประเมินภายหลัง ไม่รวมใน features
- metadata: ที่มาของแถว สถานะความพร้อม และตัวบ่งชี้คุณภาพ

KDDTest-21 เป็นชุดย่อยสำหรับประเมินผล จึงทำเครื่องหมายแถวที่ตรงกันใน KDDTest+ แทนการเพิ่มแถวซ้ำ


## 1. เตรียม path และรายชื่อไฟล์

ค้นหา project root จากตำแหน่ง notebook แทนการเขียน path เฉพาะเครื่อง จากนั้นกำหนดโฟลเดอร์ raw, cleansing และ harmonized ให้เห็นชัดเจน


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data-pipeline" / "NSL-KDD" / "download.py").is_file():
            return candidate
    raise FileNotFoundError("หา project root ไม่พบจากตำแหน่งที่เปิด notebook")

PROJECT_ROOT = find_project_root(Path.cwd())
PIPELINE_DIR = PROJECT_ROOT / "data-pipeline"
DATASET_DIR = PIPELINE_DIR / "NSL-KDD"
RAW_DIR = DATASET_DIR / "raw"
CLEAN_DIR = DATASET_DIR / "output" / "cleansing"
HARMONIZED_DIR = DATASET_DIR / "output" / "harmonized"
HARMONIZED_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Clean input:", CLEAN_DIR)
print("Harmonized output:", HARMONIZED_DIR)


## 2. ตรวจ checksum และจำนวนแถวก่อนแปลง

เทียบ raw กับ manifest ที่สร้างไว้ตอน cleansing และตรวจจำนวนแถวของไฟล์สะอาดกับไฟล์ดิบ การตรวจนี้ทำให้ orig_row_id อ้างกลับไปยังลำดับต้นทางได้อย่างปลอดภัย หากไฟล์ถูกเปลี่ยนหรือจำนวนแถวไม่ตรง notebook จะหยุด ไม่จับคู่ด้วยการเดา


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# ใช้ชนิดไฟล์เดียวกับ manifest ของ pipeline เพื่อให้ Build กลางตรวจสอบต่อได้
raw_paths = sorted(
    path for path in RAW_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in {".csv", ".gz", ".txt"}
)
if not raw_paths:
    raise FileNotFoundError(f"ไม่พบไฟล์ raw ใน {RAW_DIR}")

current_manifest = pd.DataFrame([
    {
        "file": path.relative_to(RAW_DIR).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in raw_paths
], columns=["file", "bytes", "sha256"])

manifest_path = CLEAN_DIR / "source_manifest.csv"
if not manifest_path.is_file():
    raise FileNotFoundError("ไม่พบ source_manifest.csv กรุณารัน cleansing.ipynb ใหม่ก่อน")

saved_manifest = pd.read_csv(manifest_path).sort_values("file").reset_index(drop=True)
current_manifest = current_manifest.sort_values("file").reset_index(drop=True)
if saved_manifest["file"].tolist() != current_manifest["file"].tolist():
    raise ValueError("รายชื่อ raw files เปลี่ยนหลัง cleansing; กรุณารัน cleansing.ipynb ใหม่")
if saved_manifest["sha256"].tolist() != current_manifest["sha256"].tolist():
    raise ValueError("checksum ของ raw เปลี่ยนหลัง cleansing; กรุณารัน cleansing.ipynb ใหม่")

clean_files = {
    "KDDTrain": "KDDTrain_clean.csv",
    "KDDTest": "KDDTest_clean.csv",
    "KDDTest-21": "KDDTest-21_clean.csv",
}
raw_files = {
    "KDDTrain": "KDDTrain+.txt",
    "KDDTest": "KDDTest+.txt",
    "KDDTest-21": "KDDTest-21.txt",
}
clean_data = {}
for split_name, clean_name in clean_files.items():
    clean_path = CLEAN_DIR / clean_name
    raw_path = RAW_DIR / raw_files[split_name]
    if not clean_path.is_file() or not raw_path.is_file():
        raise FileNotFoundError(f"ไฟล์ของ {split_name} ไม่ครบ: {clean_path} / {raw_path}")

    clean_frame = pd.read_csv(clean_path)
    raw_rows = sum(1 for _ in raw_path.open(encoding="utf-8"))
    if len(clean_frame) != raw_rows:
        raise ValueError(
            f"{split_name}: raw มี {raw_rows:,} แถว แต่ clean มี {len(clean_frame):,} แถว"
        )
    clean_data[split_name] = clean_frame

# ตรวจว่าไฟล์ Train 20% เป็นแถวจาก Train เต็ม ไม่ใช่ข้อมูลอิสระที่ควรนำเข้าซ้ำ
full_train_path = RAW_DIR / "KDDTrain+.txt"
reduced_train_path = RAW_DIR / "KDDTrain+_20Percent.txt"
if full_train_path.is_file() and reduced_train_path.is_file():
    full_train_rows = set(full_train_path.read_text(encoding="utf-8").splitlines())
    reduced_train_rows = reduced_train_path.read_text(encoding="utf-8").splitlines()
    unmatched_rows = [row for row in reduced_train_rows if row not in full_train_rows]
    if unmatched_rows:
        raise ValueError(
            f"พบ {len(unmatched_rows):,} แถวใน KDDTrain+_20Percent ที่ไม่อยู่ใน KDDTrain+"
        )
    print(
        "ตรวจ subset KDDTrain+_20Percent:",
        f"{len(reduced_train_rows):,} แถวอยู่ใน KDDTrain+ ทั้งหมด",
    )

print("ตรวจจำนวนแถว:", {name: len(frame) for name, frame in clean_data.items()})
display(current_manifest)


## 3. สร้าง features, labels และ metadata

การแปลงใช้ชื่อคอลัมน์จริงของ NSL-KDD เช่น src_bytes และ protocol_type แล้วเปลี่ยนเป็นชื่อกลาง ตัวอย่างเช่น src_bytes → bytes_forward ส่วน label และ attack_family จะแยกเก็บใน labels ไม่ปะปนกับ model features


In [ ]:
# ตรวจว่า KDDTest-21 เป็นแถวที่อยู่ใน KDDTest+ จริง โดยไม่สร้างข้อมูลซ้ำ
def row_hashes(frame: pd.DataFrame) -> pd.Series:
    return pd.util.hash_pandas_object(
        frame.astype("string"), index=False
    ).astype("uint64")

test_hashes = row_hashes(clean_data["KDDTest"])
test21_hashes = row_hashes(clean_data["KDDTest-21"])
if test_hashes.duplicated().any() or test21_hashes.duplicated().any():
    raise ValueError("พบแถวซ้ำใน Test หรือ Test-21 ทำให้จับคู่แบบหนึ่งต่อหนึ่งไม่ได้")

test21_hash_set = set(test21_hashes.tolist())
is_test21_row = test_hashes.isin(test21_hash_set)
if int(is_test21_row.sum()) != len(clean_data["KDDTest-21"]):
    raise ValueError("KDDTest-21 จับคู่กับ KDDTest+ ได้ไม่ครบ จึงไม่ทำเครื่องหมาย evaluation view")

print(
    f"KDDTest-21 ที่จับคู่ใน KDDTest+: {int(is_test21_row.sum()):,} "
    f"จาก {len(clean_data['KDDTest-21']):,} แถว"
)

# รวม service และสถานะการเชื่อมต่อที่มีความหมายใกล้กัน
service_aliases = {
    "http": "http", "www": "http",
    "https": "https_ssl", "ssl": "https_ssl",
    "ftp": "ftp", "ftp_data": "ftp",
    "ssh": "ssh", "smtp": "smtp",
    "domain": "dns", "domain_u": "dns", "dns": "dns",
}
connection_states = {
    "sf": "established", "fin": "established", "con": "established",
    "s0": "no-reply", "int": "no-reply", "req": "no-reply",
    "rej": "reset", "rsto": "reset", "rstr": "reset",
    "rstos0": "reset", "rst": "reset",
}
family_names = {
    "Normal": "Normal",
    "DoS": "DoS/DDoS",
    "Probe": "Probe/Recon",
    "R2L": "Brute-force/R2L",
    "U2R": "Exploit/U2R",
}


# การโจมตีบางชนิดใน Test ไม่อยู่ใน mapping สั้นของ cleansing
# จัดเข้าตระกูลที่รู้จักจากชื่อ label เดิม โดยยังเก็บ label_raw ไว้ครบ
additional_attack_families = {
    "apache2": "DoS",
    "mailbomb": "DoS",
    "processtable": "DoS",
    "udpstorm": "DoS",
    "worm": "DoS",
    "mscan": "Probe",
    "saint": "Probe",
    "snmpguess": "R2L",
    "snmpgetattack": "R2L",
    "httptunnel": "R2L",
    "named": "R2L",
    "sendmail": "R2L",
    "xlock": "R2L",
    "xsnoop": "R2L",
    "ps": "U2R",
    "xterm": "U2R",
    "sqlattack": "U2R",
}


In [ ]:
def parse_nonnegative(values: pd.Series) -> tuple[pd.Series, pd.Series]:
    # แปลงเป็นตัวเลข และแยก flag ของค่าที่ติดลบหรือแปลงไม่ได้
    numeric = pd.to_numeric(values, errors="coerce").astype("float64")
    invalid = values.notna() & (
        numeric.isna() | ~np.isfinite(numeric) | numeric.lt(0)
    )
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    numeric.loc[numeric < 0] = np.nan
    return numeric, invalid.astype(bool)


def optional_number(frame: pd.DataFrame, column: str) -> pd.Series:
    # คืนตัวเลขถ้ามีคอลัมน์ มิฉะนั้นเก็บ missing ไว้ ไม่แทนเป็นศูนย์
    if column not in frame:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    numeric = pd.to_numeric(frame[column], errors="coerce").astype("float64")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    numeric.loc[numeric < 0] = np.nan
    return numeric


def build_nsl_tables(frame: pd.DataFrame, split_name: str,
                     test21_view: pd.Series | None = None):
    frame = frame.reset_index(drop=True).copy()

    duration, invalid_duration = parse_nonnegative(frame["duration"])
    bytes_forward, invalid_forward = parse_nonnegative(frame["src_bytes"])
    bytes_backward, invalid_backward = parse_nonnegative(frame["dst_bytes"])

    protocol = frame["protocol_type"].astype("string").str.strip().str.lower()
    protocol = protocol.replace({"6": "tcp", "17": "udp", "1": "icmp"})
    protocol = protocol.where(
        protocol.isna() | protocol.isin(["tcp", "udp", "icmp"]), "other"
    )

    service = frame["service"].astype("string").str.strip().str.lower()
    service_group = service.map(service_aliases).fillna("other")
    service_group = service_group.where(service.notna()).astype("string")

    state = frame["flag"].astype("string").str.strip().str.lower()
    connection_group = state.map(connection_states).fillna("other")
    connection_group = connection_group.where(state.notna()).astype("string")

    record_ids = [
        f"NSL-KDD:{split_name}:{row_number:09d}"
        for row_number in range(len(frame))
    ]
    original_labels = frame["label"].astype("string").str.strip().str.lower()
    source_family = frame["attack_family"].astype("string").str.strip()
    attack_category = source_family.map(family_names)
    additional_families = original_labels.map(additional_attack_families)
    additional_categories = additional_families.map(family_names)
    attack_category = attack_category.fillna(additional_categories).fillna("Unknown")

    labels = pd.DataFrame({
        "record_id": record_ids,
        "label_raw": original_labels,
        "label_source_category": source_family,
        "attack_category": attack_category.astype("string"),
        "label_binary": pd.to_numeric(frame["is_attack"], errors="raise").astype("int8"),
    })

    features = pd.DataFrame({
        "record_id": record_ids,
        "duration": duration,
        "protocol": protocol,
        "bytes_forward": bytes_forward,
        "bytes_backward": bytes_backward,
        "service_group": service_group,
        "conn_state_group": connection_group,
        "packets_forward": optional_number(frame, "spkts"),
        "packets_backward": optional_number(frame, "dpkts"),
        "mean_packet_size": pd.Series(np.nan, index=frame.index, dtype="float64"),
        "ttl_forward": optional_number(frame, "sttl"),
        "ttl_backward": optional_number(frame, "dttl"),
        "tcp_window_forward": optional_number(frame, "swin"),
        "tcp_window_backward": optional_number(frame, "dwin"),
        "nsl_hot": optional_number(frame, "hot"),
        "nsl_logged_in": optional_number(frame, "logged_in"),
        "nsl_failed_logins": optional_number(frame, "num_failed_logins"),
        "nsl_root_shell": optional_number(frame, "root_shell"),
        "nsl_count_2s": optional_number(frame, "count"),
        "nsl_srv_count_2s": optional_number(frame, "srv_count"),
        "nsl_same_srv_rate_2s": optional_number(frame, "same_srv_rate"),
        "nsl_serror_rate_2s": optional_number(frame, "serror_rate"),
    })

    row_numbers = range(len(frame))
    metadata = pd.DataFrame({
        "record_id": record_ids,
        "source_dataset": "NSL-KDD",
        "orig_row_id": [
            f"{split_name}:{row_number:09d}" for row_number in row_numbers
        ],
        "source_split": "train" if split_name == "KDDTrain" else "test",
        "capture_time": pd.Series(pd.NA, index=frame.index, dtype="string"),
        "split_group": pd.Series(pd.NA, index=frame.index, dtype="string"),
        "dedup_flag": False,
        "sample_weight": 1.0,
        "has_duration": features["duration"].notna(),
        "invalid_duration": invalid_duration,
        "has_protocol": features["protocol"].notna(),
        "has_directional_bytes": features[
            ["bytes_forward", "bytes_backward"]
        ].notna().all(axis=1),
        "invalid_bytes_forward": invalid_forward,
        "invalid_bytes_backward": invalid_backward,
        "has_capture_time": False,
        "has_service": features["service_group"].notna(),
        "has_connection_state": features["conn_state_group"].notna(),
        "has_packet_stats": features[
            ["packets_forward", "packets_backward", "mean_packet_size"]
        ].notna().any(axis=1),
        "has_tcp_details": features[
            ["ttl_forward", "ttl_backward", "tcp_window_forward", "tcp_window_backward"]
        ].notna().any(axis=1),
        "has_nsl_content": features[
            ["nsl_hot", "nsl_logged_in", "nsl_failed_logins", "nsl_root_shell"]
        ].notna().any(axis=1),
        "has_nsl_window": features[
            ["nsl_count_2s", "nsl_srv_count_2s", "nsl_same_srv_rate_2s", "nsl_serror_rate_2s"]
        ].notna().any(axis=1),
        "is_kddtest21_view": (
            test21_view.to_numpy(dtype=bool)
            if test21_view is not None
            else np.zeros(len(frame), dtype=bool)
        ),
    })
    return features, labels, metadata


## 4. ตรวจตารางก่อนบันทึก

ยืนยันว่า record_id ไม่ซ้ำและเรียงตรงกันทั้งสามตาราง รวมทั้งตรวจว่าคอลัมน์ label หรือ metadata ไม่หลุดเข้า features


In [ ]:
feature_parts, label_parts, metadata_parts = [], [], []
for split_name in ("KDDTrain", "KDDTest"):
    test21_flags = is_test21_row if split_name == "KDDTest" else None
    feature_table, label_table, metadata_table = build_nsl_tables(
        clean_data[split_name], split_name, test21_flags
    )
    feature_parts.append(feature_table)
    label_parts.append(label_table)
    metadata_parts.append(metadata_table)

features = pd.concat(feature_parts, ignore_index=True)
labels = pd.concat(label_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)

# ตรวจการเชื่อมสามตารางก่อนเขียนไฟล์
for table_name, table in {
    "features": features, "labels": labels, "metadata": metadata
}.items():
    if table["record_id"].isna().any() or table["record_id"].duplicated().any():
        raise ValueError(f"{table_name}: record_id หายหรือซ้ำ")

if not features["record_id"].equals(labels["record_id"]):
    raise ValueError("record_id ใน features และ labels เรียงไม่ตรงกัน")
if not features["record_id"].equals(metadata["record_id"]):
    raise ValueError("record_id ใน features และ metadata เรียงไม่ตรงกัน")

forbidden = {
    "label", "label_raw", "attack_category", "label_binary", "is_attack",
    "source_dataset", "orig_row_id", "capture_time", "split_group",
    "sample_weight", "dedup_flag", "difficulty",
}
leaked_columns = forbidden.intersection(features.columns)
if leaked_columns:
    raise ValueError(f"พบ label/metadata ใน features: {sorted(leaked_columns)}")

print("จำนวนแถวในแต่ละตาราง:", {
    "features": len(features), "labels": len(labels), "metadata": len(metadata)
})
print("จำนวน model feature:", len(features.columns) - 1, "(ไม่รวม record_id)")
display(labels["attack_category"].value_counts().rename_axis("attack_category").to_frame("rows"))


## 5. บันทึกผลและรายงานที่มา\n\nเขียน CSV แยกสามตาราง พร้อมรายงานจำนวนแถว ความพร้อมของ feature และ provenance ซึ่งบอกเวอร์ชัน contract กับ checksum ของ raw ที่ใช้

In [ ]:
tables = {
    "features": features,
    "labels": labels,
    "metadata": metadata,
}
for table_name, table in tables.items():
    temporary_path = HARMONIZED_DIR / f".{table_name}.partial.csv"
    final_path = HARMONIZED_DIR / f"{table_name}.csv"
    table.to_csv(temporary_path, index=False, encoding="utf-8")
    temporary_path.replace(final_path)

harmonization_report = pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in tables.items()
])
harmonization_report.to_csv(
    HARMONIZED_DIR / "harmonization_report.csv", index=False
)

availability_rows = []
for column in features.columns:
    if column == "record_id":
        continue
    available = int(features[column].notna().sum())
    availability_rows.append({
        "source_dataset": "NSL-KDD",
        "feature": column,
        "rows_available": available,
        "rows_total": len(features),
        "availability_percent": 100 * available / len(features) if len(features) else 0,
    })
pd.DataFrame(availability_rows).to_csv(
    HARMONIZED_DIR / "feature_availability.csv", index=False
)

quality_report = pd.DataFrame([{
    "source_dataset": "NSL-KDD",
    "rows": len(metadata),
    "duplicate_rows": int(metadata["dedup_flag"].sum()),
    "invalid_duration": int(metadata["invalid_duration"].sum()),
    "invalid_bytes_forward": int(metadata["invalid_bytes_forward"].sum()),
    "invalid_bytes_backward": int(metadata["invalid_bytes_backward"].sum()),
}])
quality_report.to_csv(HARMONIZED_DIR / "harmonization_quality.csv", index=False)

current_manifest.to_csv(HARMONIZED_DIR / "source_manifest.csv", index=False)
registry = json.loads(
    (PIPELINE_DIR / "common" / "dataset_registry.json").read_text(encoding="utf-8")
)
contract = json.loads(
    (PIPELINE_DIR / "common" / "feature_contract.json").read_text(encoding="utf-8")
)
provenance = {
    "dataset": "NSL-KDD",
    "source_version": registry["datasets"]["NSL-KDD"]["version"],
    "registry_version": registry["registry_version"],
    "feature_contract_version": contract["contract_version"],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "row_count": len(features),
    "source_files": current_manifest.to_dict(orient="records"),
}
(HARMONIZED_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(harmonization_report)
print("ผล Harmonization:", HARMONIZED_DIR)
